In [ ]:
spark.sql("SHOW TABLES") \
    .filter("lower(tableName) LIKE '%zone%'") \
    .show(truncate=False)

In [ ]:
df_source = spark.table("silver_taxi_zones")

df_source.printSchema()
df_source.show(5, truncate=False)

print("Total rows:", df_source.count())

In [ ]:
from pyspark.sql.functions import col, count

df_source.groupBy("LocationID") \
    .agg(count("*").alias("cnt")) \
    .filter(col("cnt") > 1) \
    .show()


print("NULL business keys:",
      df_source.filter(col("LocationID").isNull()).count())

In [ ]:
from pyspark.sql.functions import sha2, to_json, struct

tracked_columns = ["Borough", "Zone", "service_zone"]

df_hashed = df_source.withColumn(
    "row_hash",
    sha2(to_json(struct(*tracked_columns)), 256)
)

df_hashed.show(5, truncate=False)

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.orderBy("LocationID")

df_initial = df_hashed.withColumn(
    "zone_sk",
    row_number().over(window_spec).cast("long")
)

df_initial.show(5, truncate=False)

In [ ]:
from pyspark.sql.functions import lit, current_timestamp

df_initial = df_initial \
    .withColumn("effective_from", current_timestamp()) \
    .withColumn("effective_to", lit(None).cast("timestamp")) \
    .withColumn("is_current", lit(True))

In [ ]:
target_table = "gold_dim_zone_scd2"

if not spark.catalog.tableExists(target_table):
    df_initial.write \
        .format("delta") \
        .mode("errorifexists") \
        .saveAsTable(target_table)

    print("Initial load completed")
else:
    print("Table already exists. Initial load skipped")

In [ ]:
df_target = spark.table(target_table)

df_target.select(
    "zone_sk",
    "LocationID",
    "Zone",
    "effective_from",
    "effective_to",
    "is_current"
).show(5, truncate=False)

print("Total records:", df_target.count())

print("Current records:",
      df_target.filter(col("is_current") == True).count())

print("Historical records:",
      df_target.filter(col("is_current") == False).count())

In [ ]:
from pyspark.sql.functions import when, col, lit

df_incoming = df_source.withColumn(
    "Zone",
    when(
        col("LocationID") == 1,
        lit("Newark Airport Terminal")
    ).otherwise(col("Zone"))
)

df_incoming.filter(col("LocationID") == 1).show()

In [ ]:
df_incoming = df_incoming.withColumn(
    "row_hash",
    sha2(to_json(struct(*tracked_columns)), 256)
)

df_incoming.filter(
    col("LocationID") == 1
).show(truncate=False)

In [ ]:
df_current = spark.table(target_table).filter(
    col("is_current") == True
)

df_changed = df_incoming.alias("s").join(
    df_current.alias("t"),
    col("s.LocationID") == col("t.LocationID"),
    "inner"
).filter(
    col("s.row_hash") != col("t.row_hash")
)

In [ ]:
df_changed.select(
    col("s.LocationID"),
    col("t.Zone").alias("old_zone"),
    col("s.Zone").alias("new_zone"),
    col("t.zone_sk").alias("old_sk")
).show(truncate=False)

print("Changed records:", df_changed.count())

In [ ]:
from pyspark.sql.functions import max as spark_max

max_sk = spark.table(target_table) \
    .agg(spark_max("zone_sk")) \
    .first()[0] or 0

print("Maximum surrogate key:", max_sk)

In [ ]:
df_updates = df_changed.select(
    "s.*",
    col("t.zone_sk").alias("old_sk")
)

df_new_keys = df_incoming.join(
    df_current.select("LocationID"),
    "LocationID",
    "left_anti"
).withColumn("old_sk", lit(None).cast("long"))

df_versions = df_updates.unionByName(df_new_keys)

In [ ]:
df_versions = df_versions.withColumn(
    "zone_sk",
    (lit(max_sk) + row_number().over(
        Window.orderBy("LocationID")
    )).cast("long")
).withColumn(
    "effective_from", current_timestamp()
).withColumn(
    "effective_to", lit(None).cast("timestamp")
).withColumn(
    "is_current", lit(True)
)

df_versions = df_versions.localCheckpoint(eager=True)

In [ ]:
df_versions.select(
    "LocationID",
    "Zone",
    "old_sk",
    "zone_sk",
    "is_current"
).show(truncate=False)

In [ ]:
df_expire = df_versions \
    .filter(col("old_sk").isNotNull()) \
    .withColumn("merge_sk", col("old_sk")) \
    .withColumn("action", lit("expire"))

df_insert = df_versions \
    .withColumn("merge_sk", col("zone_sk")) \
    .withColumn("action", lit("insert"))

df_staged = df_expire.unionByName(df_insert)

df_staged.select(
    "LocationID",
    "old_sk",
    "zone_sk",
    "merge_sk",
    "action"
).show()

In [ ]:
from delta.tables import DeltaTable

delta_target = DeltaTable.forName(spark, target_table)

delta_target.alias("t").merge(
    df_staged.alias("s"),
    "t.zone_sk = s.merge_sk"
).whenMatchedUpdate(
    condition="s.action = 'expire' AND t.is_current = true",
    set={
        "effective_to": "s.effective_from",
        "is_current": "false"
    }
).whenNotMatchedInsert(
    condition="s.action = 'insert'",
    values={
        "zone_sk": "s.zone_sk",
        "LocationID": "s.LocationID",
        "Borough": "s.Borough",
        "Zone": "s.Zone",
        "service_zone": "s.service_zone",
        "row_hash": "s.row_hash",
        "effective_from": "s.effective_from",
        "effective_to": "s.effective_to",
        "is_current": "s.is_current"
    }
).execute()

In [ ]:
df_result = spark.table(target_table)

df_result.filter(col("LocationID") == 1) \
    .select(
        "zone_sk",
        "LocationID",
        "Zone",
        "effective_from",
        "effective_to",
        "is_current"
    ).orderBy("zone_sk").show(truncate=False)

print("Total records:", df_result.count())
print("Current records:", df_result.filter(col("is_current")).count())
print("Historical records:", df_result.filter(~col("is_current")).count())

In [ ]:
df_before = spark.table(target_table).localCheckpoint(eager=True)

print("Total records:", df_before.count())
print("Current records:", df_before.filter(col("is_current")).count())
print("Historical records:", df_before.filter(~col("is_current")).count())

In [ ]:
df_before = spark.table(target_table).localCheckpoint(eager=True)

print("Before:", df_before.count())

In [ ]:
df_current = spark.table(target_table).filter(
    col("is_current") == True
)

df_changed = df_incoming.alias("s").join(
    df_current.alias("t"),
    col("s.LocationID") == col("t.LocationID"),
    "inner"
).filter(
    col("s.row_hash") != col("t.row_hash")
)

print("Changed records:", df_changed.count())

In [ ]:
df_new_keys = df_incoming.join(
    df_current.select("LocationID"),
    "LocationID",
    "left_anti"
)

print("New business keys:", df_new_keys.count())

In [ ]:
df_updates = df_changed.select(
    "s.*",
    col("t.zone_sk").alias("old_sk")
)

df_new_keys = df_new_keys.withColumn(
    "old_sk", lit(None).cast("long")
)

df_versions = df_updates.unionByName(df_new_keys)

print("Versions to insert:", df_versions.count())

In [ ]:
df_after = spark.table(target_table)

print("Before:", df_before.count())
print("After:", df_after.count())

print("Added or modified:",
      df_after.exceptAll(df_before).count())

print("Removed or modified:",
      df_before.exceptAll(df_after).count())